In [ ]:
import os
import torch

print("GPU VERIFICATION:")
device = torch.device("cuda" if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 6) else "cpu")
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if device.type == 'cuda':
    print(f"SUCCESS: Active GPU detected - {torch.cuda.get_device_name(0)} ({num_gpus} GPU(s) available)")
else:
    print("WARNING: No compatible GPU detected! Pipeline will run on CPU.")
print()

repo_dir = "/kaggle/working/zerocross-ai"
if not os.path.exists(repo_dir):
    !git clone https://github.com/muhammad-hassaan-aiml/zerocross-ai.git {repo_dir}

os.chdir(repo_dir)
!chmod +x build_kaggle.sh
!./build_kaggle.sh

In [ ]:
import os
import shutil

input_dir = "/kaggle/input/"
working_models_dir = "/kaggle/working/models"
os.makedirs(working_models_dir, exist_ok=True)

found_previous = False
if os.path.exists(input_dir):
    for root, dirs, files in os.walk(input_dir):
        if "best_model.pth" in files:
            print(f"Found previous session data in {root}!")
            print("Copying to working directory to resume training...")
            for file in files:
                if file.endswith((".pth", ".pt", ".csv", ".json")):
                    shutil.copy(os.path.join(root, file), working_models_dir)
            found_previous = True
            break

if not found_previous:
    print("No previous dataset attached. Starting a fresh session.")
else:
    print("Resume data loaded successfully!")

In [ ]:
import json, os, torch

state_path = "/kaggle/working/models/pipeline_state.json"
model_path = "/kaggle/working/models/best_model.pth"

total_iterations = 0
if os.path.exists(state_path):
    total_iterations = json.load(open(state_path)).get("total_iterations", 0)

if total_iterations == 0 and os.path.exists(model_path):
    ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    if isinstance(ckpt, dict):
        total_iterations = ckpt.get("iteration", 0)
    print("(state file missing/zero -- falling back to checkpoint's embedded iteration)")

next_iter = total_iterations + 1
if next_iter <= 100:
    lr = 0.001
elif next_iter <= 250:
    lr = 0.0005
else:
    lr = 0.0001

print(f"total_iterations (resume point): {total_iterations}")
print(f"next iteration will be:          {next_iter}")
print(f"learning rate that implies:       {lr}\n")

In [ ]:
!python python/pipeline.py \
    --iterations 100 \
    --concurrent-games 400 \
    --games-per-iteration 500 \
    --mcts-sims 200 \
    --eval-games 100 \
    --eval-sims 200 \
    --batch-size 2048 \
    --max-rejections 5 \
    --num-res-blocks 6 \
    --num-channels 128

In [ ]:
import json
print(json.load(open("/kaggle/working/models/pipeline_state.json")))